# Introduction to GeoPandas

This quick tutorial introduces the key concepts and basic features of GeoPandas.

## Concepts

GeoPandas, as the name suggests, extends the popular data science library [pandas](https://pandas.pydata.org) by adding support for geospatial data.

The core data structure in GeoPandas is the `geopandas.GeoDataFrame`, that can store geometry columns and perform spatial operations. Therefore, your `GeoDataFrame` is a combination of traditional data (numerical, boolean, text etc.), and a column with geometries (points, polygons etc.).

![geodataframe schema](https://raw.githubusercontent.com/geopandas/geopandas/8279cc33bf46dcf23bbe8cf92179951e673bea68/doc/source/_static/dataframe.svg)

Each `GeoDataFrame` can contain any geometry type and has a `crs` attribute, which stores information about the projection (CRS stands for Coordinate Reference System).

`geopandas` also provides simple visualization capacity that you can make a quick map.

Now let's see how to use `geopandas` with some real data.


## Installation


`geopandas`is not automatically installed on your computer. To install it there are two approaches:
1) Go to Anaconda Navigator -> Environments (left panel) -> base -> select All in the dropdown -> search for geopandas -> Apply -> Apply

2) Go to Anaconda Prompt or Terminal -> type `conda install geopandas`

Once it is successful, restart this notebook.

In [ ]:
import geopandas as gpd #usually people use gpd as alias

## Reading files

`geopandas` can read in popular GIS data files including shapefiles, GeoJSON, GeoPackage, etc. You can read it using `geopandas.read_file()` function, which automatically detects the filetype and creates a GeoDataFrame.

So here I put some files in our course Github repository:

That you can fetch directly from an URL.

In [ ]:
url = "https://raw.githubusercontent.com/Ziqi-Li/GIS5103/fall26/data/earthquake_2.5_month.geojson"

earthquakes = gpd.read_file(url)

A quick view of the table.

In [ ]:
earthquakes.head()

A quick map view of the data can be done by using the `.plot()` function to your `GeoDataFrame`.

In [ ]:
earthquakes.plot()

For the second data you can download a world country shapefile from [here](https://hub.arcgis.com/datasets/esri::world-countries-generalized/explore?location=-0.755783%2C0.000000%2C2.03). Click Download - > GEOJSON or Shaplefile format. You can then copy and paste to the same directory as this notebook. 

Here, for demostration purpose, I have uploaded a copy to the course github site. And you can directly read in the data from there.


In [ ]:
#url = "https://raw.githubusercontent.com/Ziqi-Li/GIS5103/fall26/data/countries.geojson"

local_geojson_path = "World_Countries_(Generalized)_9029012925078512962.geojson"

#local_shp_folder_path = "World_Countries_(Generalized)_2402777631520798174"

In [ ]:
countries = gpd.read_file(local_geojson_path)

In [ ]:
countries.plot()

In [ ]:
countries.shape

## Coordinate Reference Systems

Spatial data should always have Coordinate Reference Systems (CRS) information. The CRS tells GeoPandas where the coordinates of the geometries are located on the earth’s surface. 
- In some cases, the CRS is geographic, which means that the coordinates are in latitude and longitude. In those cases, its CRS is WGS84, with the authority code `EPSG:4326` (2D) or `EPSG:4979` (3D). 
- In other cases, the CRS is projected, which means that the coordinates are in linear unit such as in meters. A popular CRS is what Google Maps and other online maps use which is `EPSG:3857`. Or the CRS that is mostly often used for making US maps such as `ESRI:102039` for which you will see a curved north border.
- When you have multiple datasets, and it is important to make sure they are in the same CRS (aligning) before making analysis or maps.
- [epsg.io](https://epsg.io) is a comprehensive database for looking up specific CRS and its code.






In [ ]:
countries.crs

In [ ]:
earthquakes.crs

The two datasets are both in WGS84 with lat and long, but they are with different EPSG codes.

You can transform CRS from one to another using `.to_crs()` function. The function can take EPSG code as a string.

In [ ]:
countries = countries.to_crs("EPSG:3857")

In [ ]:
countries.crs

In [ ]:
countries.plot()

You can also convert one's CRS by using the other's CRS, for example:


In [ ]:
#earthquakes = earthquakes.to_crs("EPSG:3857")

earthquakes = earthquakes.to_crs(countries.crs)

In [ ]:
earthquakes.crs

## Simple attributes and functions

Now we have our `GeoDataFrame` and can start working with its geometry.

### Measuring area

To measure the area of each polygon, access the `GeoDataFrame.area` attribute, which returns a column. Note: in most of the cases, you want your data to be in a projected CRS before calculating area or distance.


In [ ]:
countries.area # the unit here is square meters

You can also create a new column in the `GeoDataFrame` called `area_km2` to store the calculated area in km2 unit.

In [ ]:
countries["area_km2"] = countries.area/1000/1000 # do a simple math to change to square km2

countries

Let's find out which country has the largest/smallest area? Have a guess before running the command?

Here we can call the `.sort_values()` function and it takes two important parameters:
- the column that we are using to sort. Here we use `area_km2`.
- whether it is sorted ascendingly (let `ascending=True`) or descendingly (let `ascending=False`).

In [ ]:
countries.sort_values('area_km2', ascending=False)

Does the ranking make any sense, if not, why?

### Getting polygon boundary and centroid

To get the boundary of each polygon (LineString), access the `GeoDataFrame.boundary`:

In [ ]:
countries.boundary.plot()

Similarly we can get centroids of countries from the country polygons.

In [ ]:
countries.centroid.plot()

Or to create a circle buffer of each location using `.buffer` and specify a distance.

We can do this for the earthquake data and create a 500,000m buffer zone to show the affected area. 

In [ ]:
earthquakes.buffer(500000).plot()

## Making maps

GeoPandas can also plot maps, so we can check how the geometries appear in space. To plot the active geometry, call `GeoDataFrame.plot()`. To color code by another column, pass in that column as the first argument. In the example below, we plot the active geometry column and color code by the `"area_km2"` column. We also want to show a legend (`legend=True`).

In [ ]:
countries.plot(column="area_km2", legend=True)

In [ ]:
earthquakes.columns

Or to make color to represent the magnitude of each earthquake.

In [ ]:
earthquakes.plot(column='mag',legend=True)

You can change the color scale by specifying `cmap='Oranges'`

Other supported color scales can be found [here](https://matplotlib.org/stable/users/explain/colors/colormaps.html)


In [ ]:
earthquakes.plot(column='mag',legend=True,cmap="Oranges_r")

More often we need to overlay multiple layers on top of each other. Here we need to put them on the same `axis`.

First, create the first layer and named it as `ax`. Then plot the second layer, and within the `plot()` function, write down `ax=ax`.

You can also set some level of transparency by including `alpha=0.2` in the plot function, and to change the default blue color into red color `color='red'`.

For listed colors you can refer to this [webpage](https://matplotlib.org/stable/gallery/color/named_colors.html)

Alternatively, you can use hex code colors such as `color = '#FF0000'`.

In [ ]:
earthquake_buffer = earthquakes.centroid.buffer(500000)

In [ ]:
ax = countries.plot(color='wheat')



earthquake_buffer.plot(ax=ax, color='red',alpha=0.2)



earthquakes.plot(ax=ax, color="black", markersize=1)

## Geometric relationships
The most commonly used geometric operation is spatial join:

- `GeoDataFrame.sjoin`

A spatial join can merge two `GeoDataFrame`s, one left, one right based on a `predicate`. The predicate can be, for example, `within`,`intersects`, `contains`, etc., for different operations.

![spatial join schema](https://datavisdotblog.files.wordpress.com/2022/01/spatial-joins-header.png?w=1204)



The results of the spatial join will be a new `GeoDataFrame` with all information from both the `GeoDataFrame` tables that meet the speficied `predicate` condition.

Here, if the interest is to count how many earthquakes fall within each country, then we will be joining the country GeoDataFrame with the earthquakes GeoDataFrame with `predicate="contains"`. For point-polygon overlay, you can also use `intersects`. And you may use other predictaes for other tasks.

In fact, counting points in polygon is a complicated task but it can be broken down into several steps:

### Step 1

In [ ]:
joined = gpd.sjoin(countries, earthquakes, 
                   predicate="contains", how='inner')

joined

The above returns every single possible pair of country-earthquake if any earthquake falls within any country. This means that:
- A country with multiple earthquakes will appear as multiple rows
- An earthquake that is not contained by any country are dropped.
- This behaviour is called an `inner` join, which is the default of `.sjoin()`. You can also use `how=left` or `how=right`, if you want to retain all the left or right records even the predicate fails. See [here](https://geopandas.org/en/stable/docs/reference/api/geopandas.sjoin.html) for more details.

### Step 2

Grouping rows by polygon ID (here I use `COUNTRY`) and count how many rows each country has (with `.size()`). The result will be a single column pandas.Series with COUNTRY name as the index. 

In [ ]:
counts = joined.groupby("COUNTRY").size()

counts

Next we need to join the result with our original dataframe. To do so, first we reset the index of the above Series so that it will be a DataFrame with COUNTRY as a seperate column

In [ ]:
counts = counts.reset_index()

counts

You can then merge this result back to your countries GeoDataFrame

In [ ]:
countries_merged = countries.merge(counts,on="COUNTRY", how="left")

countries_merged


In [ ]:
#rename the '0' as a real column name
countries_merged = countries_merged.rename(columns={0:'n_points'})

countries_merged#.head()

Then you can make a map showing the distribution of earthquake counts. The numbers are very skewed so the map doesn't look nice apart from showing US has the most earthquakes.

Also you may notice that some countries are dropped because they don't have any earthquakes.

In [ ]:
countries_merged.plot(column='n_points', legend=True)

There are two methods to fix this:
- to specify a style for the missing countries by passing a dictionary of style parameters into the `missing_kwds` parameter in `.plot()`
- or to replace NAN with 0 so that you can use normal `.plot()`

In [ ]:
countries_merged.plot(column='n_points',legend=True, 
                      missing_kwds={"color":"grey"})

In [ ]:
countries_merged

In [ ]:
countries_merged["n_points"] = countries_merged["n_points"].fillna(0)


countries_merged.plot(column='n_points', legend=True)

Other spatial join such as involving lines will follow pretty much the same logic. Essentially, you will need to first perform joining, then grouping by and using some aggregation functions.

### Creating GeoDataFrame from coodinates

Sometimes, most often if you have point location data, they are not in a spatial format, but rather in a normal table such as in csv. If you want to make it into a GeoDataFrame you can then following the below.

In [ ]:
import pandas as pd

nyc_wifi_url = "https://raw.githubusercontent.com/Ziqi-Li/GIS5103/fall26/data/nyc-wi-fi-hotspot-locations.csv"

nyc_wifi = pd.read_csv(nyc_wifi_url)

In [ ]:
nyc_wifi.head()

We will use the Latitude and Longitude columns as the geometry of our GeoDataFrame. And if it is in lat-long, the CRS will be 4326.

In [ ]:
geometry = gpd.points_from_xy(nyc_wifi.Longitude, nyc_wifi.Latitude)

nyc_wifi_gdf = gpd.GeoDataFrame(nyc_wifi, geometry=geometry, 
                                crs="EPSG:4326")

In [ ]:
nyc_wifi_gdf.plot()

In [ ]:
nyc_wifi_gdf.iloc[:, 5: 8]

## Exporting GeoDataFrame

You can use `your_data_frame.to_file()` to export your files into `GeoJSON` or `shp` shapefile formats.

In [ ]:
nyc_wifi_gdf.to_file("nyc_wifi_copy.geojson")

In [ ]:
gpd.read_file("nyc_wifi_copy.geojson").plot()

In [ ]:
nyc_wifi_gdf.to_file("nyc_wifi_copy.shp")

In [ ]:
gpd.read_file("nyc_wifi_copy.shp").plot()